# Three Body Reader — Chapter Pipeline (Qwen c+d)

Variant of the pipeline where Qwen produces *both* (c) word glosses and (d) sentence
translation in a single call, so the two rows are guaranteed to agree on idioms and
phrasing (e.g. 心急如焚 renders consistently in both rows).

Runtime: **GPU → T4** (Runtime > Change runtime type).

## 1. Upload EPUB

In [ ]:
from google.colab import files
uploaded  = files.upload()
EPUB_PATH = next(iter(uploaded))
print(f'Using: {EPUB_PATH}')

## 2. Install dependencies

In [ ]:
%%bash
apt-get install -qq zstd pciutils
curl -fsSL https://ollama.com/install.sh | sh 2>&1 | tail -5
pip install -q pypinyin jieba

## 3. Start Ollama and pull model

In [ ]:
import subprocess, time, urllib.request, json

MODEL = 'qwen2.5:14b'  # change to qwen2.5:7b if you hit VRAM limits

subprocess.Popen(['ollama', 'serve'],
                 stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

for i in range(30):
    try:
        urllib.request.urlopen('http://localhost:11434/api/tags', timeout=2)
        print('Ollama is running.')
        break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError('Ollama did not start within 30 s.')

print(f'Pulling {MODEL} ...')
subprocess.run(['ollama', 'pull', MODEL], check=True)
print('Model ready.')

## 4. Configuration

In [ ]:
CHAPTER_NUM = 1
OUTPUT_JS   = f'chapter{CHAPTER_NUM}.js'

## 5. Pipeline

In [ ]:
import re, gzip, zipfile
from pathlib import Path
from pypinyin import pinyin as get_pinyin, Style
import jieba

CEDICT_URL   = 'https://www.mdbg.net/chinese/export/cedict/cedict_1_0_ts_utf-8_mdbg.txt.gz'
CEDICT_CACHE = Path('/root/.cache/cedict/cedict.txt')
HANZI_RE     = re.compile(r'[一-鿿㐀-䶿豈-﫿]')
PUNCT_SET    = set('，。！？；：、""\'\'""\'\'「」（）【】《》〈〉·—…\t\n ')

DIGIT_PINYIN = {'0':'líng','1':'yī','2':'èr','3':'sān','4':'sì','5':'wǔ','6':'liù','7':'qī','8':'bā','9':'jiǔ'}

ANALYZE_PROMPT = """\
You are analyzing a sentence from the literary science-fiction novel 《三体》 (The Three-Body Problem) by Liu Cixin.

Given the sentence and its word segmentation, return a JSON object with exactly two keys:
- "glosses": an array of {{"seg": <word>, "gloss": <1-4 word English gloss>}} objects,
  one per word segment, in the same order as the input.
  You MUST include every segment — do not skip, merge, or reorder any, including
  grammatical particles. Use these fixed glosses for common particles:
    的 (possessive/attributive) → [poss.]
    了 (perfective aspect) → [perf.]
    着 (progressive aspect) → [prog.]
    过 (experiential aspect) → [exp.]
    地 (adverbial marker) → [adv.]
    得 (resultative marker) → [res.]
  The gloss value is always a JSON string in double quotes — never an unquoted value.
  Example: {{"seg": "的", "gloss": "[poss.]"}} — the brackets are part of the string.
  The glosses must be consistent with the translation (same rendering of idioms and imagery).
- "translation": a single literary English sentence, vivid and atmospheric.
  Preserve Chinese idioms (成语) as evocative images (e.g. 心急如焚 → "heart burning with anxiety").

Word segments ({n} total): {segments}
Sentence: {sentence}

Respond with valid JSON only. No markdown, no explanation.
"""


# ── CC-CEDICT (fallback only) ───────────────────────────────────────────────

def load_cedict():
    if not CEDICT_CACHE.exists():
        print('Downloading CC-CEDICT...')
        CEDICT_CACHE.parent.mkdir(parents=True, exist_ok=True)
        with urllib.request.urlopen(CEDICT_URL) as r:
            CEDICT_CACHE.write_bytes(gzip.decompress(r.read()))
    cedict = {}
    for line in CEDICT_CACHE.read_text('utf-8').splitlines():
        if line.startswith('#'):
            continue
        m = re.match(r'\S+ (\S+) \[[^\]]+\] /(.+)/', line)
        if not m:
            continue
        simplified, raw_defs = m.group(1), m.group(2).split('/')
        gloss = raw_defs[0]
        for d in raw_defs:
            if not re.match(r'(CL:|variant|see |abbr\.|surname )', d):
                gloss = d
                break
        cedict[simplified] = gloss[:50]
    print(f'  {len(cedict):,} entries loaded (fallback).')
    return cedict


# ── EPUB ───────────────────────────────────────────────────────────────────

def find_attr(tag_str, attr):
    m = re.search(rf'\b{attr}="([^"]+)"', tag_str)
    return m.group(1) if m else None

def parse_epub(epub_path, chapter_num):
    with zipfile.ZipFile(epub_path) as zf:
        container = zf.read('META-INF/container.xml').decode('utf-8')
        opf_path  = re.search(r'full-path="([^"]+\.opf)"', container).group(1)
        opf_dir   = str(Path(opf_path).parent)
        opf       = zf.read(opf_path).decode('utf-8')

        def full(href):
            return (opf_dir + '/' + href) if opf_dir != '.' else href

        spine_ids  = re.findall(r'<itemref\s+idref="([^"]+)"', opf)
        id_to_href = {}
        for tag in re.findall(r'<item\b[^>]*/>', opf):
            id_m   = re.search(r'\bid="([^"]+)"', tag)
            href_m = re.search(r'\bhref="([^"]+)"', tag)
            if id_m and href_m:
                id_to_href[id_m.group(1)] = href_m.group(1)
        spine_hrefs = [full(id_to_href[i]) for i in spine_ids if i in id_to_href]

        ncx_item = re.search(r'<item\b[^>]*application/x-dtbncx\+xml[^>]*/>', opf)
        if not ncx_item:
            ncx_item = re.search(r'<item\b[^>]*application/x-dtbncx\+xml[^>]*>', opf)
        if not ncx_item:
            raise RuntimeError('Could not find NCX item in OPF.')
        ncx_href = find_attr(ncx_item.group(0), 'href')
        ncx = zf.read(full(ncx_href)).decode('utf-8')

        navpoints = re.findall(
            r'<navPoint\b[^>]*>.*?<text>([^<]*)</text>.*?<content\s+src="([^"#]+)',
            ncx, re.DOTALL
        )
        chapter_files = {}
        for label, src in navpoints:
            m = re.match(r'^(\d+)\.', label.strip())
            if m:
                chapter_files[int(m.group(1))] = Path(src.strip()).name

        if chapter_num not in chapter_files:
            raise ValueError(f'Chapter {chapter_num} not found. Available: {sorted(chapter_files)}')

        start_name = chapter_files[chapter_num]
        end_name   = chapter_files.get(chapter_num + 1)

        in_chapter, chapter_hrefs = False, []
        for href in spine_hrefs:
            name = Path(href).name
            if name == start_name:
                in_chapter = True
            if in_chapter:
                if end_name and name == end_name:
                    break
                chapter_hrefs.append(href)

        paragraphs = []
        for href in chapter_hrefs:
            html = zf.read(href).decode('utf-8', errors='replace')
            for raw in re.findall(r'<p[^>]*>(.*?)</p>', html, re.DOTALL):
                text = re.sub(r'<[^>]+>', '', raw)
                text = re.sub(r'\s+', '', text)
                if not text or text == '※※※':
                    continue
                if re.match(r'^\d+[\.。]', text):
                    continue
                paragraphs.append(text)
    return paragraphs


# ── NLP ────────────────────────────────────────────────────────────────────

def split_sentences(text):
    parts = re.split(r'(?<=[。！？])', text)
    return [p.strip() for p in parts if p.strip()]

def is_punct_token(token):
    return all(c in PUNCT_SET for c in token)

def tokenize(text):
    """Return (words, segments) where words includes punct dicts and
    segments is the ordered list of non-punct token strings for Qwen."""
    words, segments = [], []
    for token in jieba.cut(text):
        if not token or not token.strip():
            continue
        if is_punct_token(token):
            for ch in token:
                if ch.strip():
                    words.append({'punct': True, 'chars': [ch]})
        else:
            chars = list(token)
            raw   = get_pinyin(token, style=Style.TONE)
            if len(raw) == len(chars):
                pinyins = [p[0] for p in raw]
            else:
                # pypinyin collapsed multi-char token (e.g. numbers); expand per-char
                pinyins = [get_pinyin(ch, style=Style.TONE)[0][0] for ch in chars]
            pinyins = [DIGIT_PINYIN.get(p, p) for p in pinyins]
            words.append({'chars': chars, 'pinyins': pinyins, 'gloss': ''})
            segments.append(token)
    return words, segments


# ── Qwen: analyze sentence (c + d together) ────────────────────────────────

def _call_ollama(prompt, timeout=300):
    payload = json.dumps({
        'model':   MODEL,
        'prompt':  prompt,
        'stream':  False,
        'options': {'temperature': 0},
    }).encode()
    req = urllib.request.Request(
        'http://localhost:11434/api/generate',
        data=payload,
        headers={'Content-Type': 'application/json'},
    )
    with urllib.request.urlopen(req, timeout=timeout) as r:
        return json.loads(r.read())['response'].strip()

def _parse_json(text):
    text = re.sub(r'^```(?:json)?\s*', '', text.strip())
    text = re.sub(r'\s*```$',          '', text.strip())
    text = text.strip()
    # Replace fullwidth comma used as JSON structural separator
    text = re.sub(r'，(?=\s*")', ',', text)
    return json.loads(text)

def _align_glosses(pairs, segments, cedict):
    """Greedy left-to-right alignment of Qwen's {seg, gloss} pairs to jieba segments.
    Handles small count mismatches by looking ahead when exact position fails.
    Returns (glosses, miss_count)."""
    glosses = []
    misses  = 0
    j = 0
    for seg in segments:
        if j < len(pairs) and pairs[j].get('seg') == seg:
            glosses.append(pairs[j].get('gloss', ''))
            j += 1
        else:
            # Look ahead up to 3 positions (handles Qwen inserting extra pairs)
            found = False
            for skip in range(1, 4):
                if j + skip < len(pairs) and pairs[j + skip].get('seg') == seg:
                    glosses.append(pairs[j + skip].get('gloss', ''))
                    j = j + skip + 1
                    found = True
                    break
            if not found:
                glosses.append(cedict.get(seg, cedict.get(seg[0] if seg else '', '')))
                misses += 1
    return glosses, misses


def analyze_sentence(sent_text, words, segments, cedict):
    """Call Qwen once to get glosses (c) + translation (d) consistently.
    Fills word['gloss'] in-place and returns the translation string."""
    prompt = ANALYZE_PROMPT.format(
        n=len(segments),
        segments=json.dumps(segments, ensure_ascii=False),
        sentence=sent_text,
    )
    raw = _call_ollama(prompt)
    try:
        data        = _parse_json(raw)
        pairs       = data['glosses']
        translation = data['translation']
        glosses, misses = _align_glosses(pairs, segments, cedict)
        if misses:
            print(f'    [{misses}/{len(segments)} glosses fell back to CC-CEDICT]')
    except Exception as e:
        print(f'    [JSON parse failed: {e}; falling back to CC-CEDICT]')
        glosses     = [cedict.get(seg, cedict.get(seg[0], '')) for seg in segments]
        translation = raw

    gi = 0
    for w in words:
        if not w.get('punct'):
            w['gloss'] = glosses[gi] if gi < len(glosses) else cedict.get(''.join(w['chars']), '')
            gi += 1

    return translation


# ── Warm-up ────────────────────────────────────────────────────────────────

def warmup():
    print(f'Warming up {MODEL} (loading into VRAM)...')
    result = _call_ollama('Say hello in one word.', timeout=300)
    print(f'  Ready. Test output: "{result}"')


# ── JS output ──────────────────────────────────────────────────────────────

def _js_str(s):
    return s.replace('\\', '\\\\').replace("'", "\\'")

def render_js(paragraphs_data, var_name):
    lines = [f'window.{var_name} = [']
    for para in paragraphs_data:
        lines.append('  { sentences: [')
        for sent in para:
            translation = sent['translation'].replace('`', '\\`')
            lines.append('    {')
            lines.append(f'      translation: `{translation}`,')
            lines.append('      words: [')
            for w in sent['words']:
                if w.get('punct'):
                    lines.append(f"        {{ punct: true, chars: ['{_js_str(w['chars'][0])}'] }},")
                else:
                    chars   = ', '.join(f"'{_js_str(c)}'" for c in w['chars'])
                    pinyins = ', '.join(f"'{_js_str(p)}'" for p in w['pinyins'])
                    gloss   = _js_str(w['gloss'])
                    lines.append(f"        {{ chars: [{chars}], pinyins: [{pinyins}], gloss: '{gloss}' }},")
            lines.append('      ],')
            lines.append('    },')
        lines.append('  ] },')
    lines.append('];')
    return '\n'.join(lines)


print('Pipeline functions defined.')


## 6. Run

In [ ]:
cedict = load_cedict()

print(f'Extracting chapter {CHAPTER_NUM} from {EPUB_PATH}...')
raw_paragraphs = parse_epub(EPUB_PATH, CHAPTER_NUM)[:3]
print(f'  {len(raw_paragraphs)} paragraphs (limited for testing).')

warmup()

paragraphs_data = []
sent_count = 0

for pi, para_text in enumerate(raw_paragraphs):
    sentences  = split_sentences(para_text)
    para_sents = []
    for si, sent_text in enumerate(sentences):
        sent_count += 1
        print(f'  [{pi+1}/{len(raw_paragraphs)}] s{si+1}: {sent_text[:50]}', flush=True)
        words, segments = tokenize(sent_text)
        translation     = analyze_sentence(sent_text, words, segments, cedict)
        print(f'    → {translation[:70]}', flush=True)
        para_sents.append({'translation': translation, 'words': words})
    paragraphs_data.append(para_sents)

var_name = f'CHAPTER{CHAPTER_NUM}'
Path(OUTPUT_JS).write_text(render_js(paragraphs_data, var_name), encoding='utf-8')
print(f'\nWrote {OUTPUT_JS} ({len(raw_paragraphs)} paragraphs, {sent_count} sentences).')


## 7. Download result

In [ ]:
from google.colab import files
files.download(OUTPUT_JS)